# Core 3 · Gold
Owned by: Data platform team. Reads from other domains only through shortcuts.

In [ ]:
# Domain: core · owner: Data platform team
# Attach this notebook to the lakehouse "Core" in the workspace "Solvi Core".
DOMAIN = "core"
LANDING_PATH = f"Files/landing/{DOMAIN}"   # upload this domain's CSV files here
TABLE_FORMAT = "delta"
LOCAL_PREFIX = ""                            # local testing sets this to the domain name
AS_OF = "2026-10-31"                         # demo data ends here; use the current date with live data

from pyspark.sql import functions as F, Window as W

def write_table(df, name):
    (df.write.format(TABLE_FORMAT).mode("overwrite").option("overwriteSchema", "true").saveAsTable(LOCAL_PREFIX + name))
    print(f"wrote {name:<30} {df.count():>9,} rows")

def read_table(name):                         # a table this domain owns
    return spark.read.table(LOCAL_PREFIX + name)

def read_shared(domain, name):                # another domain's table, through a OneLake shortcut named domain + name
    return spark.read.table(f"{domain}{name}")

Shared master data. Everyone reads these through shortcuts; nobody copies them.

In [ ]:
dim_date = (spark.sql("SELECT explode(sequence(to_date('2012-01-01'), to_date('2027-12-31'), interval 1 day)) AS date")
  .select(F.date_format("date","yyyyMMdd").cast("int").alias("date_key"), "date", F.year("date").alias("year"),
          F.quarter("date").alias("quarter"), F.month("date").alias("month_no"), F.date_format("date","MMM").alias("month"),
          F.date_format("date","yyyy-MM").alias("year_month"), F.weekofyear("date").alias("iso_week")))
write_table(dim_date, "dimdate")
site = read_table("silversite")
write_table(site, "dimsite")
write_table(read_table("silvercustomer").join(site, "site_id"), "dimcustomer")
write_table(read_table("silverproduct"), "dimproduct")
write_table(read_table("silverweatherdaily"), "factweatherdaily")
write_table(read_table("silverweatherforecast"), "factweatherforecast")